#Carga de Silver a Golden

In [0]:
# CONFIGURACIÓN DE PARÁMETROS

dbutils.widgets.text("catalog_name", "banking_dev", "Catálogo")
dbutils.widgets.text("storage_account", "adlsprojectbank12", "Storage Account")
dbutils.widgets.dropdown("load_mode", "FULL", ["FULL"], "Modo de carga")
catalog_name = dbutils.widgets.get("catalog_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
load_mode = dbutils.widgets.get("load_mode").strip().upper()

if not catalog_name or not storage_account or load_mode != "FULL":
    raise ValueError("Parámetros inválidos. Esta versión solo admite FULL.")
print(catalog_name, storage_account, load_mode)

In [0]:
# CONFIGURACIÓN DE FUENTES Y DESTINOS

source_names = [
    "customers", "accounts", "cards", "transactions", "loans", "payments",
    "sbs_creditos_sector_economico", "sbs_depositos_mn", "sbs_depositos_me",
    "sbs_morosidad", "fraud_transactions"
]
target_names = [
    "dim_customer", "dim_account", "dim_card", "dim_date",
    "fact_transactions", "fact_loans", "fact_payments",
    "agg_sbs_creditos", "agg_sbs_depositos", "agg_sbs_morosidad", "agg_fraud_daily"
]
source_tables = {n: f"{catalog_name}.silver.{n}" for n in source_names}
target_tables = {n: f"{catalog_name}.golden.{n}" for n in target_names}
target_paths = {n: f"abfss://golden@{storage_account}.dfs.core.windows.net/{n}" for n in target_names}
print(f"Fuentes: {len(source_tables)} | Destinos: {len(target_tables)}")

In [0]:
# LECTURA DE SILVER & FUNCIONES

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

silver = {}
for name, full_name in source_tables.items():
    if not spark.catalog.tableExists(full_name):
        raise ValueError(f"No existe la fuente Silver: {full_name}")
    silver[name] = spark.table(full_name)

def cast_to_target_schema(df, target_table):
    schema = spark.table(target_table).schema
    expected = [f.name for f in schema]
    missing = [c for c in expected if c not in df.columns]
    extra = [c for c in df.columns if c not in expected]
    if missing or extra:
        raise ValueError(f"{target_table}: columnas faltantes={missing}; adicionales={extra}")
    return df.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in schema])

def assert_unique_key(df, keys, label):
    null_cond = None
    for c in keys:
        cond = F.col(c).isNull()
        null_cond = cond if null_cond is None else null_cond | cond
    nulls = df.filter(null_cond).count()
    dups = df.groupBy(*keys).count().filter(F.col("count") > 1).count()
    if nulls or dups:
        raise ValueError(f"{label}: nulos={nulls:,}; claves duplicadas={dups:,}; clave={keys}")
    print(f"OK: clave única {label} {keys}")

print("Fuentes Silver leídas.")
for n, df in silver.items():
    print(f"{n}: {df.count():,}")

#### Dimensiones: clientes, cuentas y tarjetas

In [0]:
dim_customer = (
    silver["customers"]
    .select("customer_id", "customer_segment", "city", "customer_status")
    .withColumn("customer_key", F.row_number().over(Window.orderBy("customer_id")).cast("long"))
    .select("customer_key", "customer_id", "customer_segment", "city", "customer_status")
)

dim_account = (
    silver["accounts"]
    .select("account_id", "customer_id", "account_type", "currency", "account_status")
    .withColumn("account_key", F.row_number().over(Window.orderBy("account_id")).cast("long"))
    .select("account_key", "account_id", "customer_id", "account_type", "currency", "account_status")
)

dim_card = (
    silver["cards"]
    .select("card_id", "account_id", "card_type", "card_brand", "card_status")
    .withColumn("card_key", F.row_number().over(Window.orderBy("card_id")).cast("long"))
    .select("card_key", "card_id", "account_id", "card_type", "card_brand", "card_status")
)

golden = {"dim_customer": dim_customer, "dim_account": dim_account, "dim_card": dim_card}
print("Dimensiones construidas")

####Dimensiones: calendario

In [0]:
# Calendario continuo desde la fecha mínima hasta la máxima de transacciones, desembolsos, vencimientos y pagos

date_dfs = [
    silver["transactions"].select(F.to_date("transaction_date").alias("d")),
    silver["loans"].select(F.col("disbursement_date").alias("d")),
    silver["loans"].select(F.col("due_date").alias("d")),
    silver["payments"].select(F.col("payment_date").alias("d"))
]

all_dates = date_dfs[0]

for ddf in date_dfs[1:]:
    all_dates = all_dates.unionByName(ddf)
bounds = all_dates.agg(F.min("d").alias("min_d"), F.max("d").alias("max_d")).first()
if bounds["min_d"] is None or bounds["max_d"] is None:
    raise ValueError("No hay fechas válidas para dim_date")

dim_date = (
    spark.range(1)
    .select(F.explode(F.sequence(
        F.lit(bounds["min_d"]).cast("date"),
        F.lit(bounds["max_d"]).cast("date"),
        F.expr("INTERVAL 1 DAY")
    )).alias("full_date"))
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date").cast("int"))
    .withColumn("quarter", F.quarter("full_date").cast("int"))
    .withColumn("month", F.month("full_date").cast("int"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date").cast("int"))
    .withColumn("day_of_week", F.dayofweek("full_date").cast("int"))
    .select("date_key", "full_date", "year", "quarter", "month", "month_name", "day", "day_of_week")
)

golden["dim_date"] = dim_date
print(f"dim_date: {bounds['min_d']} a {bounds['max_d']}")

####Tablas de hechos

In [0]:
# Fact transactions: una fila por transaction_id, customer_id se obtiene desde accounts
t = silver["transactions"].alias("t")
a = silver["accounts"].select("account_id", "customer_id").alias("a")

fact_transactions = (
    t.join(a, F.col("t.account_id") == F.col("a.account_id"), "left")
    .withColumn("date_key", F.date_format(F.to_date("t.transaction_date"), "yyyyMMdd").cast("int"))
    .select(
        F.col("t.transaction_id").alias("transaction_id"), "date_key",
        F.col("a.customer_id").alias("customer_id"),
        F.col("t.account_id").alias("account_id"), F.col("t.card_id").alias("card_id"),
        F.col("t.amount").alias("amount"), F.col("t.currency").alias("currency"),
        F.col("t.transaction_type").alias("transaction_type"),
        F.col("t.channel").alias("channel"),
        F.col("t.transaction_status").alias("transaction_status"),
        F.col("t.is_fraud").alias("is_fraud")
    )
)

fact_loans = silver["loans"].select(
    "loan_id", "customer_id", "principal_amount", "interest_rate", "term_months", "loan_status"
)

fact_payments = silver["payments"].select(
    "payment_id", "loan_id", "payment_date", "payment_amount",
    "principal_paid", "interest_paid", "payment_status"
)

golden.update({
    "fact_transactions": fact_transactions,
    "fact_loans": fact_loans,
    "fact_payments": fact_payments
})
print("Tres tablas de hechos construidas.")

####Tablas de agregaciones SBS

In [0]:
# Créditos: la fuente ya tiene grano único por fecha, concepto y nivel
agg_sbs_creditos = silver["sbs_creditos_sector_economico"].select(
    "fecha_corte", "concepto", "nivel_jerarquia",
    F.col("total_creditos_directos_miles_soles").alias("total_creditos_miles_soles"),
    F.col("distribucion_creditos_directos_pct").alias("distribucion_pct")
)

# Depósitos: unir MN y ME y agregar por fecha, empresa y moneda
depositos = (
    silver["sbs_depositos_mn"].select(
        "fecha_corte", "empresa", "moneda", "numero_cuentas", "saldo_final_miles_soles"
    ).unionByName(
        silver["sbs_depositos_me"].select(
            "fecha_corte", "empresa", "moneda", "numero_cuentas", "saldo_final_miles_soles"
        )
    )
)

agg_sbs_depositos = depositos.groupBy("fecha_corte", "empresa", "moneda").agg(
    F.sum("numero_cuentas").cast("long").alias("numero_cuentas"),
    F.sum("saldo_final_miles_soles").cast(T.DecimalType(18, 2)).alias("saldo_final_miles_soles")
)

# Morosidad: conservar valor fuente, incluidos NULL; no promediar
agg_sbs_morosidad = silver["sbs_morosidad"].select(
    "fecha_corte", "empresa_bancaria", "concepto_credito", "morosidad_pct"
)

golden.update({
    "agg_sbs_creditos": agg_sbs_creditos,
    "agg_sbs_depositos": agg_sbs_depositos,
    "agg_sbs_morosidad": agg_sbs_morosidad
})
print("Agregaciones SBS construidas.")

####Tablas de agregaciones Fraud

In [0]:
# El dataset no tiene fecha real de transacción, transaction_date es temporal

fraud = silver["fraud_transactions"].withColumn(
    "transaction_date", F.to_date("ingestion_date")
)

agg_fraud_daily = fraud.groupBy("transaction_date").agg(
    F.count(F.lit(1)).cast("long").alias("total_transactions"),
    F.sum(F.when(F.col("Class") == 1, 1).otherwise(0)).cast("long").alias("fraud_transactions"),
    F.sum(F.when(F.col("Class") == 0, 1).otherwise(0)).cast("long").alias("legitimate_transactions"),
    F.sum(F.when(F.col("Class") == 1, F.col("Amount")).otherwise(F.lit(0)))
     .cast(T.DecimalType(18, 2)).alias("fraud_amount"),
    (F.sum(F.when(F.col("Class") == 1, 1).otherwise(0)) * F.lit(100.0) /
     F.count(F.lit(1))).cast(T.DecimalType(10, 4)).alias("fraud_rate")
)

golden["agg_fraud_daily"] = agg_fraud_daily
print("agg_fraud_daily construida usando fecha de ingesta como fecha de referencia.")

In [0]:
# VALIDAR GRANULARIDAD Y CLAVES

for table, key in [
    ("dim_customer", ["customer_id"]), ("dim_customer", ["customer_key"]),
    ("dim_account", ["account_id"]), ("dim_account", ["account_key"]),
    ("dim_card", ["card_id"]), ("dim_card", ["card_key"]),
    ("dim_date", ["full_date"]), ("dim_date", ["date_key"]),
    ("fact_transactions", ["transaction_id"]), ("fact_loans", ["loan_id"]),
    ("fact_payments", ["payment_id"]),
    ("agg_sbs_creditos", ["fecha_corte", "concepto", "nivel_jerarquia"]),
    ("agg_sbs_depositos", ["fecha_corte", "empresa", "moneda"]),
    ("agg_sbs_morosidad", ["fecha_corte", "empresa_bancaria", "concepto_credito"]),
    ("agg_fraud_daily", ["transaction_date"])
]:
    assert_unique_key(golden[table], key, table)
print("Granularidad y claves validadas.")

In [0]:
# VALIDAR INTEGRIDAD

def validate_fk(child, child_col, parent, parent_col, label, nullable=False):
    values = child.filter(F.col(child_col).isNotNull()) if nullable else child
    missing = values.select(F.col(child_col).alias("_c")).distinct().join(
        parent.select(F.col(parent_col).alias("_p")).distinct(),
        F.col("_c") == F.col("_p"), "left_anti"
    ).count()
    if missing:
        raise ValueError(f"{label}: {missing:,} valores sin correspondencia")
    print(f"OK: {label}")

ft = golden["fact_transactions"]
incomplete = ft.filter(
    F.col("customer_id").isNull() | F.col("account_id").isNull() | F.col("date_key").isNull()
).count()
if incomplete:
    raise ValueError(f"fact_transactions: {incomplete:,} filas sin cliente, cuenta o date_key")
validate_fk(ft, "customer_id", golden["dim_customer"], "customer_id", "fact_transactions.customer_id")
validate_fk(ft, "account_id", golden["dim_account"], "account_id", "fact_transactions.account_id")
validate_fk(ft, "card_id", golden["dim_card"], "card_id", "fact_transactions.card_id", nullable=True)
validate_fk(ft, "date_key", golden["dim_date"], "date_key", "fact_transactions.date_key")
validate_fk(golden["fact_loans"], "customer_id", golden["dim_customer"], "customer_id", "fact_loans.customer_id")
validate_fk(golden["fact_payments"], "loan_id", silver["loans"], "loan_id", "fact_payments.loan_id")
print("Integridad referencial validada.")

In [0]:
# VALIDAR ESQUEMAS & TABLAS DESTINO GOLDEN

# Celda 11. Validación de esquemas y destinos Golden
if set(golden) != set(target_tables):
    raise ValueError(f"Tablas construidas={sorted(golden)}; destinos={sorted(target_tables)}")

golden_ready = {}

for name, df in golden.items():
    target = target_tables[name]
    if not spark.catalog.tableExists(target):
        raise ValueError(f"No existe tabla destino: {target}")
    detail = spark.sql(f"DESCRIBE DETAIL {target}").first()
    if detail["format"].lower() != "delta":
        raise ValueError(f"El destino no es Delta: {target}")
    expected = target_paths[name].rstrip("/")
    actual = detail["location"].rstrip("/")
    if actual.lower() != expected.lower():
        raise ValueError(f"LOCATION inesperada en {target}: esperada={expected}; actual={actual}")
    golden_ready[name] = cast_to_target_schema(df, target)
    print(f"Destino validado: {target} | {actual}")
    
print("Los 11 destinos y esquemas validados.")

In [0]:
# VALIDACIÓN DE VOLÚMENES A CARGAR
for name in target_names:
    print(f"{name}: {golden_ready[name].count():,} registros")

print("Validación de volúmenes completada.")

In [0]:
# CARGA FULL A GOLDEN

for name in target_names:
    target = target_tables[name]
    golden_ready[name].write.format("delta").mode("overwrite").insertInto(target)
    print(f"Carga completada: {target}")
print("Carga FULL Silver a Golden finalizada.")

In [0]:
# VALIDAR CARGA

summary = []
for name in target_names:
    prepared = golden_ready[name].count()
    loaded = spark.table(target_tables[name]).count()
    status = "SUCCESS" if prepared == loaded else "FAILED"
    summary.append((name, prepared, loaded, status))
    if status != "SUCCESS":
        raise ValueError(f"{name}: preparado={prepared:,}; Golden={loaded:,}")
summary_df = spark.createDataFrame(
    summary, ["table_name", "prepared_count", "golden_count", "status"]
)
display(summary_df.orderBy("table_name"))
print("Las 11 tablas Golden tienen los registros esperados")

In [0]:
%sql
select * from banking_dev.golden.agg_sbs_creditos
limit 20;